#### 1. Easy ocr

In [ ]:
import cv2
import os
import numpy as np
from pdf2image import convert_from_path
from PIL import Image
import easyocr

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 이미지 전처리 함수
def preprocess_image(img):
    img_array = np.array(img)
    
    # 컬러 이미지인 경우에만 처리
    if len(img_array.shape) == 3:
        # HSV 색상 공간으로 변환
        hsv = cv2.cvtColor(img_array, cv2.COLOR_RGB2HSV)
        
        # 노란색 범위 정의 - 첨부된 이미지의 노란색에 맞게 조정
        # 밝은 노란색/황색 범위
        lower_yellow = np.array([20, 100, 180])  # 색상(H), 채도(S), 명도(V)
        upper_yellow = np.array([35, 255, 255])
        
        # 노란색 마스크 생성
        yellow_mask = cv2.inRange(hsv, lower_yellow, upper_yellow)
        
        # 마스크를 적용하여 노란색 부분을 흰색으로 변경
        result = img_array.copy()
        result[yellow_mask > 0] = [255, 255, 255]  # RGB 흰색
        
        # 그레이스케일로 변환
        gray = cv2.cvtColor(result, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_array
    
    # 이후 이진화 등 기존 처리 계속 진행
    _, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    
    # 나머지 처리 단계들...
    denoised = cv2.fastNlMeansDenoising(binary, None, 10, 7, 21)
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(denoised, -1, kernel)
    
    processed_img = Image.fromarray(sharpened)
    return processed_img

def detect_tables(image):
    """이미지에서 표 영역 감지"""
    img_array = np.array(image)

    gray = img_array
    
    # CV_8UC1 타입으로 변환
    if gray.dtype != np.uint8:
        gray = gray.astype(np.uint8)
    
    threshold = 255 - gray
    
    # 수평/수직 라인 감지
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    
    horizontal_lines = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)
    vertical_lines = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, vertical_kernel, iterations=1)
    
    # 표 테두리 감지
    table_boundaries = cv2.addWeighted(horizontal_lines, 0.5, vertical_lines, 0.5, 0.0)
    
    # 테두리에서 표 영역 찾기
    contours, _ = cv2.findContours(table_boundaries, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    table_regions = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w > 100 and h > 100:  # 작은 영역 필터링
            table_regions.append((x, y, x+w, y+h))
    
    return table_regions

def detect_table_structure_with_lines(image):
    """선 감지를 통한 표 구조 분석"""
    img_array = np.array(image)
    
    gray = img_array
    
    # CV_8UC1 타입으로 변환
    if gray.dtype != np.uint8:
        gray = gray.astype(np.uint8)
    
    # 이진화
    threshold = 255 - gray
    
    # 수평선과 수직선 감지
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (80, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 80))
    
    horizontal_lines = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    vertical_lines = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, vertical_kernel, iterations=2)
    
    # 선의 위치 추출
    h_contours, _ = cv2.findContours(horizontal_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    v_contours, _ = cv2.findContours(vertical_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 수평선 위치 (Y 좌표)
    h_positions = []
    for contour in h_contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w > 200:  # 충분히 긴 선만 고려
            h_positions.append(y + h//2)
    
    # 수직선 위치 (X 좌표)
    v_positions = []
    for contour in v_contours:
        x, y, w, h = cv2.boundingRect(contour)
        if h > 50:  # 충분히 긴 선만 고려
            v_positions.append(x + w//2)
    
    # 정렬 및 중복 제거
    h_positions = sorted(list(set(h_positions)))
    v_positions = sorted(list(set(v_positions)))
    
    # 너무 가까운 선들 병합
    h_positions = merge_close_positions(h_positions, 20)
    v_positions = merge_close_positions(v_positions, 20)
    
    return h_positions, v_positions

def merge_close_positions(positions, tolerance):
    """가까운 위치의 선들을 병합"""
    if not positions:
        return []
    
    merged = [positions[0]]
    for pos in positions[1:]:
        if pos - merged[-1] > tolerance:
            merged.append(pos)
    return merged

def parse_ocr_detection(detection):
    """OCR 감지 결과 파싱"""
    if len(detection) < 2:
        return None
    
    bbox, text = detection[0], detection[1]
    confidence = detection[2] if len(detection) > 2 else 1.0
    
    if not text or not text.strip():
        return None
    
    # bbox는 4개 점의 좌표 [(x1,y1), (x2,y2), (x3,y3), (x4,y4)]
    points = np.array(bbox, dtype=np.int32)
    x_min, y_min = points.min(axis=0)
    x_max, y_max = points.max(axis=0)
    x_center = int(np.mean(points[:, 0]))
    y_center = int(np.mean(points[:, 1]))
    
    return {
        'text': text.strip(),
        'x_center': x_center,
        'y_center': y_center,
        'x_min': x_min,
        'y_min': y_min,
        'x_max': x_max,
        'y_max': y_max,
        'bbox': bbox,
        'confidence': confidence
    }

def is_text_in_table(x_center, y_center, table_regions):
    """텍스트가 표 영역에 속하는지 확인"""
    for table_region in table_regions:
        tx1, ty1, tx2, ty2 = table_region
        if (x_center >= tx1 and x_center <= tx2 and 
            y_center >= ty1 and y_center <= ty2):
            return True
    return False

def sort_texts_within_cell(texts):
    """셀 내 텍스트 정렬 (위→아래, 왼쪽→오른쪽)"""
    if not texts:
        return []
    
    # Y 좌표로 행 그룹화 (같은 행 내에서는 X 좌표로 정렬)
    texts.sort(key=lambda x: (x['y_center'], x['x_center']))
    
    # 행 그룹 생성
    row_tolerance = 15  # 셀 내에서는 더 엄격하게
    rows = []
    current_row = [texts[0]]
    current_y = texts[0]['y_center']
    
    for text in texts[1:]:
        if abs(text['y_center'] - current_y) <= row_tolerance:
            current_row.append(text)
        else:
            rows.append(current_row)
            current_row = [text]
            current_y = text['y_center']
    
    if current_row:
        rows.append(current_row)
    
    # 각 행 내에서 X 좌표로 정렬하고 연결
    result_texts = []
    for row in rows:
        row.sort(key=lambda x: x['x_center'])
        row_text = ' '.join([t['text'].strip() for t in row if t['text'].strip()])
        if row_text:
            result_texts.append(row_text)
    
    return result_texts

def sort_easyocr_results_approach(result, image):
    """표 영역과 비표 영역을 구분해서 정렬"""
    # 표 영역 감지
    table_regions = detect_tables(image)
    
    # OCR 결과 파싱
    parsed_results = []
    for detection in result:
        parsed = parse_ocr_detection(detection)
        if parsed is None:
            continue
        
        # 표 영역에 속하는지 확인
        parsed['in_table'] = is_text_in_table(parsed['x_center'], parsed['y_center'], table_regions)
        parsed_results.append(parsed)
    
    if not parsed_results:
        return []
    
    # 표 텍스트와 일반 텍스트 분리
    table_texts = [item for item in parsed_results if item['in_table']]
    non_table_texts = [item for item in parsed_results if not item['in_table']]
    
    # 모든 텍스트를 Y 좌표 순서로 정렬하여 순서 결정
    all_texts_sorted = sorted(parsed_results, key=lambda x: x['y_center'])
    
    final_output = []
    processed_items = set()
    
    for item in all_texts_sorted:
        if id(item) in processed_items:
            continue
            
        if item['in_table']:
            # 표 영역 처리: 같은 표의 모든 텍스트를 한번에 처리
            current_table_region = None
            for table_region in table_regions:
                tx1, ty1, tx2, ty2 = table_region
                if (item['x_center'] >= tx1 and item['x_center'] <= tx2 and
                    item['y_center'] >= ty1 and item['y_center'] <= ty2):
                    current_table_region = table_region
                    break
            
            if current_table_region:
                tx1, ty1, tx2, ty2 = current_table_region
                # 같은 표 영역의 모든 텍스트 수집
                region_texts = [t for t in table_texts 
                               if (t['x_center'] >= tx1 and t['x_center'] <= tx2 and
                                   t['y_center'] >= ty1 and t['y_center'] <= ty2)]
                
                if region_texts:
                    # 표 구조 분석 (선 감지)
                    h_positions, v_positions = detect_table_structure_with_lines(image)
                    
                    # 표 영역에 해당하는 선만 필터링
                    table_h_positions = [h for h in h_positions if ty1 <= h <= ty2]
                    table_v_positions = [v for v in v_positions if tx1 <= v <= tx2]
                    
                    # 표 경계 추가
                    if ty1 not in table_h_positions:
                        table_h_positions.append(ty1)
                    if ty2 not in table_h_positions:
                        table_h_positions.append(ty2)
                    if tx1 not in table_v_positions:
                        table_v_positions.append(tx1)
                    if tx2 not in table_v_positions:
                        table_v_positions.append(tx2)
                    
                    table_h_positions.sort()
                    table_v_positions.sort()
                    
                    # 셀 그리드 생성
                    cells = []
                    for i in range(len(table_h_positions) - 1):
                        for j in range(len(table_v_positions) - 1):
                            y1, y2 = table_h_positions[i], table_h_positions[i + 1]
                            x1, x2 = table_v_positions[j], table_v_positions[j + 1]
                            
                            if (x2 - x1) > 30 and (y2 - y1) > 20:
                                cells.append({
                                    'row': i,
                                    'col': j,
                                    'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                                    'texts': []
                                })
                    
                    # 텍스트를 셀에 할당
                    for text_item in region_texts:
                        best_cell = None
                        best_overlap = 0
                        
                        text_x1, text_y1 = text_item['x_min'], text_item['y_min']
                        text_x2, text_y2 = text_item['x_max'], text_item['y_max']
                        text_area = (text_x2 - text_x1) * (text_y2 - text_y1)
                        
                        for cell in cells:
                            overlap_x1 = max(text_x1, cell['x1'])
                            overlap_y1 = max(text_y1, cell['y1'])
                            overlap_x2 = min(text_x2, cell['x2'])
                            overlap_y2 = min(text_y2, cell['y2'])
                            
                            if overlap_x1 < overlap_x2 and overlap_y1 < overlap_y2:
                                overlap_area = (overlap_x2 - overlap_x1) * (overlap_y2 - overlap_y1)
                                overlap_ratio = overlap_area / text_area if text_area > 0 else 0
                                
                                if overlap_ratio > best_overlap:
                                    best_overlap = overlap_ratio
                                    best_cell = cell
                        
                        if best_cell and best_overlap > 0.3:  # 임계값을 낮춤
                            best_cell['texts'].append(text_item)
                    
                    # 셀별로 정렬된 출력 생성
                    cells_with_text = [cell for cell in cells if cell['texts']]
                    cells_with_text.sort(key=lambda x: (x['row'], x['col']))
                    
                    # 행별로 그룹화
                    rows = {}
                    for cell in cells_with_text:
                        row_idx = cell['row']
                        if row_idx not in rows:
                            rows[row_idx] = []
                        rows[row_idx].append(cell)
                    
                    # 표 내용 추가
                    for row_idx in sorted(rows.keys()):
                        row_cells = sorted(rows[row_idx], key=lambda x: x['col'])
                        
                        row_parts = []
                        for cell in row_cells:
                            cell_texts = sort_texts_within_cell(cell['texts'])
                            if cell_texts:
                                cell_content = ' '.join(cell_texts)
                                row_parts.append(cell_content)
                        
                        if row_parts:
                            row_text = ' '.join(row_parts)
                            final_output.append(row_text)
                    
                    # 처리된 텍스트들 표시
                    for t in region_texts:
                        processed_items.add(id(t))
        else:
            # 비표 텍스트 처리: 같은 행의 모든 텍스트를 수집
            if id(item) not in processed_items:
                current_y = item['y_center']
                row_tolerance = 25
                
                # 같은 행에 속하는 모든 비표 텍스트 수집
                same_row_texts = []
                for candidate in non_table_texts:
                    if (abs(candidate['y_center'] - current_y) <= row_tolerance and 
                        id(candidate) not in processed_items):
                        same_row_texts.append(candidate)
                
                if same_row_texts:
                    # X 좌표로 정렬
                    same_row_texts.sort(key=lambda x: x['x_center'])
                    
                    # 텍스트 연결
                    row_texts = [t['text'].strip() for t in same_row_texts if t['text'].strip()]
                    if row_texts:
                        row_text = ' '.join(row_texts)
                        final_output.append(row_text)
                    
                    # 처리된 텍스트들 표시
                    for t in same_row_texts:
                        processed_items.add(id(t))
    
    return final_output

def process_pdf_to_ocr_approach(pdf_path, output_folder="./easyocr_output"):
    """PDF OCR 처리"""
    # PDF를 이미지로 변환
    images = convert_from_path(pdf_path, dpi=300)
    
    # 각 이미지 전처리
    processed_images = [preprocess_image(img) for img in images]
    
    # easyOCR 리더 초기화
    reader = easyocr.Reader(lang_list=['ko', 'en'], gpu=True)
    
    # 출력 폴더 생성
    os.makedirs(output_folder, exist_ok=True)
    
    all_texts = []
    
    # 각 페이지에 대해 OCR 수행
    for i, img in enumerate(processed_images):        
        # OCR 실행
        result = reader.readtext(
            np.array(img),
            text_threshold=0.7,
            low_text=0.4,
            slope_ths=0.1,
            width_ths=0.6,
            paragraph=False,
            link_threshold=0.3,
        )
        
        # 정렬
        sorted_page_text_list = sort_easyocr_results_approach(result, img)
        sorted_page_text = '\n'.join(sorted_page_text_list)
        all_texts.append(sorted_page_text)
        
        # 페이지별 정렬된 텍스트 출력
        print(f"\n=== 페이지 {i+1} 정렬된 텍스트 ===")
        for line in sorted_page_text_list:
            print(line)
        print("=" * 50)
    
    # 파일 저장
    pdf_filename = os.path.splitext(os.path.basename(pdf_path))[0]
    ocr_engine_name = "easyocr"
    txt_file_path = os.path.join(output_folder, f"{pdf_filename}_{ocr_engine_name}_ocr.txt")
    
    with open(txt_file_path, 'w', encoding='utf-8') as f:
        f.write('\n\n===== 페이지 구분선 =====\n\n'.join(all_texts))
    
    print(f"\n{txt_file_path}에 저장되었습니다.")
    return txt_file_path, processed_images, reader

In [ ]:
# 실행
if __name__ == "__main__":
    # pdf_path = "./termsheet/1000000424794_2. Bond Forward_장외파생금융상품 거래확인서_SC_20250204.pdf"
    pdf_path = "./termsheet/1000000019362_NH_거래확인서_20231215.pdf"
    result_path, images, reader = process_pdf_to_ocr_approach(pdf_path)

#### 2. 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_approach_sorting(image, save_path=None):
    """텍스트 정렬 과정을 시각화"""
    img_array = np.array(image)
    
    # EasyOCR 결과 가져오기
    result = reader.readtext(
        img_array,
        text_threshold=0.7,
        low_text=0.4,
        slope_ths=0.1,
        width_ths=0.6,
        paragraph=False,
        link_threshold=0.3,
    )
    
    # 결과가 없으면 스킵
    if not result:
        print("OCR 결과가 없습니다.")
        return
    
    # 표 영역 감지
    table_regions = detect_tables(image)
    
    # OCR 결과 파싱
    parsed_results = []
    for detection in result:
        parsed = parse_ocr_detection(detection)
        if parsed is None:
            continue
        
        # 표 영역에 속하는지 확인
        parsed['in_table'] = is_text_in_table(parsed['x_center'], parsed['y_center'], table_regions)
        parsed_results.append(parsed)
    
    if not parsed_results:
        print("파싱된 텍스트가 없습니다.")
        return
    
    # 시각화
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # 1. 표 영역 감지 결과
    axes[0].imshow(img_array, cmap='gray')
    axes[0].set_title('Approach: Table Detection', fontsize=14)
    axes[0].axis('off')
    
    # 표 영역 그리기
    for table_region in table_regions:
        tx1, ty1, tx2, ty2 = table_region
        table_rect = patches.Rectangle((tx1, ty1), tx2-tx1, ty2-ty1,
                                     linewidth=3, edgecolor='yellow', 
                                     facecolor='yellow', alpha=0.2)
        axes[0].add_patch(table_rect)
    
    # 텍스트 박스 그리기 (표/비표 구분)
    for i, item in enumerate(parsed_results):
        color = 'green' if item['in_table'] else 'blue'
        rect = patches.Rectangle((item['x_min'], item['y_min']), 
                               item['x_max']-item['x_min'], item['y_max']-item['y_min'],
                               linewidth=2, edgecolor=color, facecolor='none')
        axes[0].add_patch(rect)
        
        axes[0].text(item['x_min'], item['y_min']-5, f'{i+1}', 
                    fontsize=10, color=color, weight='bold')
    
    # 2. 정렬 결과
    axes[1].imshow(img_array, cmap='gray')
    axes[1].set_title('Approach: Sorted Result', fontsize=14)
    axes[1].axis('off')
    
    # 표 영역 표시
    for table_region in table_regions:
        tx1, ty1, tx2, ty2 = table_region
        table_rect = patches.Rectangle((tx1, ty1), tx2-tx1, ty2-ty1,
                                     linewidth=2, edgecolor='yellow', 
                                     facecolor='none', alpha=0.7)
        axes[1].add_patch(table_rect)
    
    # 정렬 시뮬레이션
    table_texts = [item for item in parsed_results if item['in_table']]
    non_table_texts = [item for item in parsed_results if not item['in_table']]
    all_texts_sorted = sorted(parsed_results, key=lambda x: x['y_center'])
    
    processed_items = set()
    sorted_order = 1
    colors = ['red', 'orange', 'purple', 'brown', 'pink', 'magenta', 'cyan']
    
    for item in all_texts_sorted:
        if id(item) in processed_items:
            continue
            
        if item['in_table']:
            # 표 영역 처리
            current_table_region = None
            for table_region in table_regions:
                tx1, ty1, tx2, ty2 = table_region
                if (item['x_center'] >= tx1 and item['x_center'] <= tx2 and
                    item['y_center'] >= ty1 and item['y_center'] <= ty2):
                    current_table_region = table_region
                    break
            
            if current_table_region:
                tx1, ty1, tx2, ty2 = current_table_region
                # 같은 표 영역의 모든 텍스트 수집
                region_texts = [t for t in table_texts 
                               if (t['x_center'] >= tx1 and t['x_center'] <= tx2 and
                                   t['y_center'] >= ty1 and t['y_center'] <= ty2)]
                
                if region_texts:
                    # 표 구조 분석 (선 감지)
                    h_positions, v_positions = detect_table_structure_with_lines(image)
                    
                    # 표 영역에 해당하는 선만 필터링
                    table_h_positions = [h for h in h_positions if ty1 <= h <= ty2]
                    table_v_positions = [v for v in v_positions if tx1 <= v <= tx2]
                    
                    # 표 경계 추가
                    if ty1 not in table_h_positions:
                        table_h_positions.append(ty1)
                    if ty2 not in table_h_positions:
                        table_h_positions.append(ty2)
                    if tx1 not in table_v_positions:
                        table_v_positions.append(tx1)
                    if tx2 not in table_v_positions:
                        table_v_positions.append(tx2)
                    
                    table_h_positions.sort()
                    table_v_positions.sort()
                    
                    # 구조선 그리기
                    for h_line in table_h_positions:
                        axes[1].plot([tx1, tx2], [h_line, h_line], 'cyan', linewidth=1, alpha=0.7)
                    
                    for v_line in table_v_positions:
                        axes[1].plot([v_line, v_line], [ty1, ty2], 'cyan', linewidth=1, alpha=0.7)
                    
                    # 셀 그리드 생성
                    cells = []
                    for i in range(len(table_h_positions) - 1):
                        for j in range(len(table_v_positions) - 1):
                            y1, y2 = table_h_positions[i], table_h_positions[i + 1]
                            x1, x2 = table_v_positions[j], table_v_positions[j + 1]
                            
                            if (x2 - x1) > 30 and (y2 - y1) > 20:
                                cells.append({
                                    'row': i,
                                    'col': j,
                                    'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                                    'texts': []
                                })
                    
                    # 텍스트를 셀에 할당
                    for text_item in region_texts:
                        best_cell = None
                        best_overlap = 0
                        
                        text_x1, text_y1 = text_item['x_min'], text_item['y_min']
                        text_x2, text_y2 = text_item['x_max'], text_item['y_max']
                        text_area = (text_x2 - text_x1) * (text_y2 - text_y1)
                        
                        for cell in cells:
                            overlap_x1 = max(text_x1, cell['x1'])
                            overlap_y1 = max(text_y1, cell['y1'])
                            overlap_x2 = min(text_x2, cell['x2'])
                            overlap_y2 = min(text_y2, cell['y2'])
                            
                            if overlap_x1 < overlap_x2 and overlap_y1 < overlap_y2:
                                overlap_area = (overlap_x2 - overlap_x1) * (overlap_y2 - overlap_y1)
                                overlap_ratio = overlap_area / text_area if text_area > 0 else 0
                                
                                if overlap_ratio > best_overlap:
                                    best_overlap = overlap_ratio
                                    best_cell = cell
                        
                        if best_cell and best_overlap > 0.3:
                            best_cell['texts'].append(text_item)
                    
                    # 셀별로 정렬된 순서로 박스 그리기
                    cells_with_text = [cell for cell in cells if cell['texts']]
                    cells_with_text.sort(key=lambda x: (x['row'], x['col']))
                    
                    # 행별로 그룹화하여 색상 적용
                    rows = {}
                    for cell in cells_with_text:
                        row_idx = cell['row']
                        if row_idx not in rows:
                            rows[row_idx] = []
                        rows[row_idx].append(cell)
                    
                    # 표 내용 시각화
                    for row_idx in sorted(rows.keys()):
                        row_cells = sorted(rows[row_idx], key=lambda x: x['col'])
                        color = colors[row_idx % len(colors)]
                        
                        for cell in row_cells:
                            for text_item in cell['texts']:
                                rect = patches.Rectangle((text_item['x_min'], text_item['y_min']), 
                                                       text_item['x_max']-text_item['x_min'], 
                                                       text_item['y_max']-text_item['y_min'],
                                                       linewidth=2, edgecolor=color, facecolor='none')
                                axes[1].add_patch(rect)
                                
                                # 정렬된 순서 번호
                                axes[1].text(text_item['x_min'], text_item['y_min']-5, f'{sorted_order}', 
                                            fontsize=12, color=color, weight='bold')
                                axes[1].text(text_item['x_max']-25, text_item['y_max']+15, 
                                            f'R{row_idx+1}C{cell["col"]+1}', 
                                            fontsize=8, color=color, weight='bold')
                                sorted_order += 1
                    
                    # 처리된 텍스트들 표시
                    for t in region_texts:
                        processed_items.add(id(t))
        else:
            # 비표 텍스트 처리
            if id(item) not in processed_items:
                current_y = item['y_center']
                row_tolerance = 25
                
                # 같은 행에 속하는 모든 비표 텍스트 수집
                same_row_texts = []
                for candidate in non_table_texts:
                    if (abs(candidate['y_center'] - current_y) <= row_tolerance and 
                        id(candidate) not in processed_items):
                        same_row_texts.append(candidate)
                
                if same_row_texts:
                    # X 좌표로 정렬
                    same_row_texts.sort(key=lambda x: x['x_center'])
                    
                    # 비표 텍스트 시각화
                    for text_item in same_row_texts:
                        rect = patches.Rectangle((text_item['x_min'], text_item['y_min']), 
                                               text_item['x_max']-text_item['x_min'], 
                                               text_item['y_max']-text_item['y_min'],
                                               linewidth=2, edgecolor='blue', facecolor='none')
                        axes[1].add_patch(rect)
                        
                        axes[1].text(text_item['x_min'], text_item['y_min']-5, f'{sorted_order}', 
                                    fontsize=12, color='blue', weight='bold')
                        sorted_order += 1
                    
                    # 처리된 텍스트들 표시
                    for t in same_row_texts:
                        processed_items.add(id(t))
    
    plt.tight_layout()
    
    # 범례 추가
    legend_elements = [
        patches.Patch(color='yellow', alpha=0.5, label='Table Region'),
        patches.Patch(color='green', label='Table Text (Original)'),
        patches.Patch(color='blue', label='Non-table Text'),
        patches.Patch(color='cyan', label='Table Structure Lines'),
        patches.Patch(color='red', label='Sorted Order')
    ]
    axes[1].legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.show()

def visualize_all_pages():
    """모든 페이지 시각화"""
    # 전역 변수 확인
    if 'images' not in globals() or not images:
        print("원본 이미지가 로드되지 않았습니다.")
        return
    
    if 'reader' not in globals():
        print("OCR reader가 초기화되지 않았습니다.")
        return
    
    print(f"총 {len(images)}페이지 시각화를 시작합니다.")
    
    for i, image in enumerate(images):
        print(f"\n=== 페이지 {i+1} 시각화 ===")
        
        visualize_approach_sorting(image)

# 실행
if __name__ == "__main__":    
    # 모든 페이지 시각화 실행
    visualize_all_pages()

#### 3. 일괄처리 파이프라인

In [ ]:
import os
import numpy as np
from pdf2image import convert_from_path
import easyocr

import glob
from pathlib import Path

def get_pdf_files(folder_path):
    """폴더 내 모든 PDF 파일 경로 가져오기"""
    folder = Path(folder_path)
    if not folder.exists():
        return []
    
    # pathlib만 사용하여 중복 방지
    pdf_files = list(folder.glob("*.pdf"))
    pdf_files = sorted([str(p) for p in pdf_files])
    
    return pdf_files

def process_single_pdf_to_txt(pdf_path, output_folder, reader):
    """단일 PDF 파일을 개별 텍스트 파일로 처리"""
    try:
        # PDF를 이미지로 변환
        images = convert_from_path(pdf_path, dpi=300)
        
        # 각 이미지 전처리
        processed_images = [preprocess_image(img) for img in images]
        
        all_texts = []
        
        # 각 페이지에 대해 OCR 수행
        for i, img in enumerate(processed_images):        
            print(f"    페이지 {i+1}/{len(processed_images)} 처리 중...")
            
            # OCR 실행
            result = reader.readtext(
                np.array(img),
                text_threshold=0.7,
                low_text=0.4,
                slope_ths=0.1,
                width_ths=0.6,
                paragraph=False,
                link_threshold=0.3,
            )
            
            # 정렬
            sorted_page_text_list = sort_easyocr_results_approach(result, img)
            sorted_page_text = '\n'.join(sorted_page_text_list)
            all_texts.append(sorted_page_text)
        
        # 파일 저장
        pdf_filename = os.path.splitext(os.path.basename(pdf_path))[0]
        ocr_engine_name = "easyocr"
        txt_file_path = os.path.join(output_folder, f"{pdf_filename}_{ocr_engine_name}_ocr.txt")
        
        with open(txt_file_path, 'w', encoding='utf-8') as f:
            f.write('\n\n===== 페이지 구분선 =====\n\n'.join(all_texts))
        
        return {
            'status': 'success',
            'input_path': pdf_path,
            'output_path': txt_file_path,
            'pages': len(processed_images),
            'error': None
        }
        
    except Exception as e:
        return {
            'status': 'failed',
            'input_path': pdf_path,
            'output_path': None,
            'pages': 0,
            'error': str(e)
        }

def batch_process_pdfs_to_txt(input_folder, output_folder, max_files=None, start_index=0):
    """폴더 내 PDF를 개별 텍스트 파일로 일괄 처리"""
    
    # 입력 폴더 존재 확인
    if not os.path.exists(input_folder):
        print(f"입력 폴더가 존재하지 않습니다: {input_folder}")
        return None
    
    # PDF 파일들 찾기
    all_pdf_files = get_pdf_files(input_folder)
    
    if not all_pdf_files:
        print(f"PDF 파일이 없습니다: {input_folder}")
        return None
    
    # 시작 인덱스와 최대 파일 수 적용
    if start_index >= len(all_pdf_files):
        print(f"시작 인덱스가 너무 큽니다. 전체 파일 수: {len(all_pdf_files)}")
        return None
    
    end_index = len(all_pdf_files)
    if max_files is not None:
        end_index = min(start_index + max_files, len(all_pdf_files))
    
    pdf_files = all_pdf_files[start_index:end_index]
    
    print(f"전체 PDF 파일: {len(all_pdf_files)}개")
    print(f"처리할 파일: {len(pdf_files)}개 (인덱스 {start_index+1}-{end_index})")
    
    for i, pdf_file in enumerate(pdf_files, 1):
        print(f"  {i}. {os.path.basename(pdf_file)}")
    
    # 출력 폴더 생성
    os.makedirs(output_folder, exist_ok=True)
    print(f"\n출력 폴더: {output_folder}")
    
    # 처리 결과 저장용
    results = []
    successful_count = 0
    failed_count = 0
    
    # easyOCR 리더 초기화 (한 번만)
    print("\nOCR 리더 초기화 중...")
    reader = easyocr.Reader(lang_list=['ko', 'en'], gpu=True)
    print("OCR 리더 초기화 완료")
    
    # 각 PDF 파일 처리
    for i, pdf_path in enumerate(pdf_files, 1):
        print(f"\n[{i}/{len(pdf_files)}] 처리 중: {os.path.basename(pdf_path)}")
        
        result = process_single_pdf_to_txt(pdf_path, output_folder, reader)
        results.append(result)
        
        if result['status'] == 'success':
            successful_count += 1
            print(f"  완료: {os.path.basename(result['output_path'])}")
            print(f"  페이지 수: {result['pages']}개")
        else:
            failed_count += 1
            print(f"  실패: {result['error']}")
    
    # 처리 결과 요약 출력
    print_batch_summary(results, input_folder, output_folder)
    
    return results

def print_batch_summary(results, input_folder, output_folder):
    """일괄 처리 결과 요약 출력"""
    successful_results = [r for r in results if r['status'] == 'success']
    failed_results = [r for r in results if r['status'] == 'failed']
    
    print(f"\n{'='*60}")
    print(f"일괄 처리 완료 요약")
    print(f"{'='*60}")
    print(f"입력 폴더: {input_folder}")
    print(f"출력 폴더: {output_folder}")
    print(f"전체 파일: {len(results)}개")
    print(f"성공: {len(successful_results)}개")
    print(f"실패: {len(failed_results)}개")
    
    if successful_results:
        print(f"\n성공한 파일들:")
        for result in successful_results:
            filename = os.path.basename(result['input_path'])
            output_filename = os.path.basename(result['output_path'])
            print(f"  {filename} -> {output_filename} ({result['pages']}페이지)")
    
    if failed_results:
        print(f"\n실패한 파일들:")
        for result in failed_results:
            filename = os.path.basename(result['input_path'])
            print(f"  {filename}: {result['error']}")
    
    print(f"\n생성된 텍스트 파일들:")
    if os.path.exists(output_folder):
        txt_files = [f for f in os.listdir(output_folder) if f.endswith('.txt')]
        for txt_file in sorted(txt_files):
            print(f"  {txt_file}")

def run_batch_processing_direct(input_folder="./termsheet", output_folder="./easyocr_output", 
                               max_files=None, start_index=0):
    """직접 경로 지정으로 일괄 처리 실행"""
    print("PDF 일괄 처리 시작")
    print("="*50)
    print(f"입력 폴더: {input_folder}")
    print(f"출력 폴더: {output_folder}")
    print(f"처리 개수: {max_files if max_files else '전체'}")
    print(f"시작 번호: {start_index + 1}")
    
    results = batch_process_pdfs_to_txt(input_folder, output_folder, max_files, start_index)
    return results

# 실행 방법들
if __name__ == "__main__":
    results = run_batch_processing_direct(
        input_folder="./termsheet/선도채권/", 
        output_folder="../ocr_post/ocr_output",
        max_files=100,
        start_index=0     # 첫 번째 파일부터 시작
    )